# Day 3 — Applied Lab: Voice Agents & the Real-Time Stack
### Agentic Customer Experience Specialisation · Post-lunch session (4h)

**What we're building today:** a voice agent that answers a claim-status call by actually
listening and speaking — the same tool-using discipline from Day 1/2, now wired to a real-time
audio pipeline instead of a text session. By the end you'll have assembled a streaming
STT→LLM→TTS loop and measured it against a latency budget, built a resilient STT layer that
survives a provider failure, and built a compliant call flow that discloses itself, asks for
consent, and only records once granted.

**How the six post-lunch topics map onto this notebook:**

| # | Topic | Where |
|---|---|---|
| 1 | Pipeline assembly | Lab 1 (Insurance), Steps 1–3 |
| 2 | Latency engineering | Lab 1 (Insurance), Steps 4–5 |
| 3 | Telephony | Lab 3 (Telecom), Step 8 |
| 4 | Reliability | Lab 2 (Banking), Step 6 |
| 5 | Voice eval & QA | Lab 2 (Banking), Step 7 |
| 6 | Compliance | Lab 3 (Telecom), Step 9 |

**Demo lanes:** Insurance (Lab 1 — assemble the pipeline, hit the latency budget — **today's ship
line**), Banking (Lab 2 — STT primary + fallback, WER-gated quality check), Telecom (Lab 3 —
SIP call flow with disclosure, consent, and a replayable audit trail). Same three-lane structure
as Day 2 — each topic pair gets its own vertical instead of one lane carrying all six.

**Continuity with Day 1 and Day 2:** Lab 1's claim-lookup tool is Day 2's `claim_lookup` and
Day 1's Insurance KB, given a voice — yesterday the claim id was typed, today the caller says it.
Same `claims_db`, same `CLM-1077` denial case, same discipline: ground the answer, cite the
source, never guess.

**A different verification bar than Day 1/2, and why — read this before starting:** Day 1 and 2
ran every cell against a real, authenticated Claude session — no API key needed, no hardware
needed. Voice is different: this sandbox has no microphone, no speakers, no phone line, and no
`DEEPGRAM_API_KEY` / `CARTESIA_API_KEY` / `ANTHROPIC_API_KEY`. Every lab below is explicit about
three tiers:

| Tier | Meaning |
|---|---|
| 🟢 **Tier A** | Runs for real, right now, in this notebook — no key, no hardware |
| 🟡 **Tier B** | Real code you can run with a free-tier provider key + the bundled WAV fixture — not live-run here |
| 🔴 **Tier C** | Needs a real phone line / SIP trunk — cannot run in any notebook, marked and explained [the implementation covers how a real working script would look like in real use-cases] |

This isn't a workaround — it's the actual discipline the day is teaching. A latency number you
didn't measure isn't engineered, it's guessed; a compliance gate you didn't test isn't a
guarantee, it's a hope. Tier A is where that discipline gets to be real today; Tier B and C are
named honestly instead of quietly assumed.

**Framework choice:** [Pipecat](https://docs.pipecat.ai) — its pipeline is a literal Python list
of processors, which makes every teaching beat (assembly, a latency fix, a fallback swap, a
compliance gate) a visible, inspectable object change. [LiveKit Agents](https://docs.livekit.io/agents/)
builds the same kind of agent with a different shape; it's the Day 2-style framework-literacy
appendix at the end, not the main thread.

| Marker | Meaning |
|---|---|
| ▶ Run this | A cell you execute and observe the output of |
| ✅ Checkpoint | A natural save point |
| 🔍 Try it yourself | A prompt to test with your own input |


## Setup

1. Python 3.10+, then:
   ```bash
   pip install "pipecat-ai[deepgram,cartesia,silero,anthropic,assemblyai,elevenlabs]"
   ```
2. For real (Tier B) runs, a Deepgram key, a Cartesia key, and a raw `ANTHROPIC_API_KEY` —
   **note the last one specifically**: the Claude Agent SDK used on Day 1/2 authenticates
   through the Claude Code CLI (`claude login`), but Pipecat's `AnthropicLLMService` talks to
   the Anthropic API directly and only accepts a real `ANTHROPIC_API_KEY` environment variable.
   The CLI login does not cover it — same gotcha Day 2's LangChain/LangGraph appendix flagged.
3. Nothing else — Silero VAD's ONNX model ships **bundled inside pipecat-ai itself**
   (`pipecat/audio/vad/data/silero_vad.onnx`), and Pipecat implements Cartesia/ElevenLabs/
   AssemblyAI itself over raw `aiohttp`/`websockets` rather than depending on each vendor's own
   SDK — so installing pipecat-ai's extras is the whole install, no separate vendor packages to
   chase down.

**A real trap worth naming before you hit it:** Pipecat is a fast-moving framework and its own
docs say so directly — "confidently-wrong old APIs are the #1 failure mode" for anyone (human or
AI) writing Pipecat code from memory or an old tutorial. Concretely, in the version pinned here
(1.5.0), constructing a pipeline directly via `PipelineTask` and running it with `PipelineRunner`
— the pattern in almost every older tutorial — is **deprecated**: the current names are
`PipelineWorker` and `WorkerRunner`. Every service below is also built with the current
`settings=` pattern rather than the deprecated flat keyword arguments (`model=`, `voice_id=`,
etc.) many older examples still show. This notebook was built by reading the installed package's
actual source, not from memory — worth doing the same before trusting any Pipecat snippet you
didn't just verify.

▶ **Run this** to confirm the install.


In [ ]:
# Setup — run this first.
import asyncio, json, time, wave
import numpy as np

import pipecat
print("pipecat-ai version:", pipecat.__version__ if hasattr(pipecat, "__version__") else "(see pip show)")

from pipecat.pipeline.pipeline import Pipeline
from pipecat.pipeline.worker import PipelineWorker, PipelineParams
from pipecat.workers.runner import WorkerRunner
from pipecat.audio.vad.silero import SileroVADAnalyzer
from pipecat.audio.vad.vad_analyzer import VADParams

print("Environment ready.")


---

## Lab 1 — Insurance: assemble the loop, hit the latency budget

**Ship criterion — today's ship line:** a Pipecat pipeline that answers a claim-status question
by voice — STT, a claim-lookup tool call, TTS — configured to keep voice-to-voice latency inside
a stated budget, with each stage's contribution to that budget actually measured or honestly
estimated, not assumed.

### Step 1 — The knowledge base and claims data

Identical to Day 1's `policy_chunks`/`search()` and Day 2's `claims_db` — reused verbatim, not
rebuilt. The only thing that changes today is that the customer *says* "CLM-1077" instead of
typing it, and the answer comes back as audio instead of text.


In [ ]:
policy_chunks = [
    {"id": "POL-4.2", "text": "Comprehensive coverage does not include rental vehicle "
     "reimbursement unless Rider R-12 (Rental Reimbursement) has been purchased separately."},
    {"id": "POL-4.3", "text": "Collision coverage applies to damages resulting from an accident "
     "involving the insured vehicle and does not extend to third-party rental vehicles."},
    {"id": "POL-9.1", "text": "Rider R-12 provides up to INR 1,500/day for rental vehicle costs, "
     "capped at 30 days, while the insured vehicle is under repair due to a covered claim."},
    {"id": "POL-2.5", "text": "A covered claim requires an incident report filed within 7 days "
     "of the event and, for collision claims, a repair estimate from an approved garage."},
]

def score(query: str, text: str) -> float:
    q = set(query.lower().split())
    t = set(text.lower().split())
    return len(q & t) / max(len(q), 1)

def search(query: str, top_k: int = 3):
    ranked = sorted(policy_chunks, key=lambda c: score(query, c["text"]), reverse=True)
    return ranked[:top_k]

claims_db = {
    "CLM-1000": {"policy_id": "POL-100", "status": "approved", "filed": "2026-07-14",
                 "amount": 25000},
    "CLM-1042": {"policy_id": "POL-233", "status": "under review", "filed": "2026-07-16",
                 "amount": 8000},
    "CLM-1077": {"policy_id": "POL-509", "status": "denied", "filed": "2026-07-09",
                 "amount": 15000,
                 "denial_reason": "Incident report filed after the 7-day window (POL-2.5)."},
}

print("KB and claims_db ready — same data as Day 1/2.")


### Step 2 — The claim-lookup tool, Pipecat-style

Same idea as Day 1/2's `@tool`-decorated functions, different mechanics: a Pipecat tool is a
**plain async function** — its name, typed signature, and docstring become the schema
automatically (no separate name/description/schema arguments to pass, unlike
`claude_agent_sdk.tool` or even LangChain's `@tool` decorator). It reports back through
`params.result_callback(...)`, which is what actually feeds the result into the conversation so
the LLM can speak it — there's no return value to use instead.

**Why this matters for voice specifically:** the LLM's output is *spoken*, not read — Pipecat's
own guidance is explicit that markdown, bullet points, and emoji come out as audible noise
through TTS. That constraint doesn't touch this tool (it just returns a dict), but it does shape
the system prompt in Step 4: told to answer in one flowing spoken sentence, not a bulleted list.


In [ ]:
from pipecat.services.llm_service import FunctionCallParams

async def voice_claim_lookup(params: FunctionCallParams, claim_id: str):
    """Look up the status of an existing insurance claim by its claim id.

    Args:
        claim_id: The claim id the caller stated, e.g. "CLM-1077".
    """
    record = claims_db.get(claim_id)
    if not record:
        await params.result_callback({"error": f"No claim found with id {claim_id}"})
        return
    await params.result_callback(record)

# Offline check — no pipeline, no LLM, no key. Same discipline as Day 1/2's .handler() checks:
# this proves the tool's OWN logic is correct; it says nothing about whether a real LLM will
# call it correctly or speak the result naturally — that needs the real pipeline (Tier B/C).
class _FakeResultCallback:
    def __init__(self):
        self.received = None
    async def __call__(self, result):
        self.received = result

async def _test_voice_claim_lookup():
    cb = _FakeResultCallback()
    fake_params = FunctionCallParams(
        function_name="voice_claim_lookup", tool_call_id="test-1", arguments={"claim_id": "CLM-1077"},
        llm=None, pipeline_worker=None, context=None, result_callback=cb,
    )
    await voice_claim_lookup(fake_params, claim_id="CLM-1077")
    print("CLM-1077 lookup result:", cb.received)
    assert cb.received["status"] == "denied"
    assert "POL-2.5" in cb.received["denial_reason"]

    cb2 = _FakeResultCallback()
    fake_params2 = FunctionCallParams(
        function_name="voice_claim_lookup", tool_call_id="test-2", arguments={"claim_id": "CLM-9999"},
        llm=None, pipeline_worker=None, context=None, result_callback=cb2,
    )
    await voice_claim_lookup(fake_params2, claim_id="CLM-9999")
    print("unknown claim result:", cb2.received)
    assert "error" in cb2.received

await _test_voice_claim_lookup()
print("\nOK: voice_claim_lookup's own logic is correct — no pipeline involved yet.")


### Step 3 — A pipeline configured for accuracy, not latency

Three individually reasonable-sounding choices, made without thinking about the latency budget,
that collectively blow it. Each is a **real, documented** Pipecat setting — this isn't a
contrived strawman:

- **`interim_results=False`** on the STT service. Deepgram normally streams *partial* transcripts
  as the caller talks; turning this off means the STT stage only hands anything downstream once
  the transcript is fully finalized — simpler to reason about, and it adds the caller's entire
  trailing pause to the critical path before the LLM even starts.
- **`text_aggregation_mode=TextAggregationMode.SENTENCE`** on the TTS service. Pipecat's own
  docstring for this setting is direct about the cost: *"Buffer text until sentence boundaries
  are detected before synthesis. Produces more natural speech but adds latency (~200-300ms per
  sentence)."* Reasonable for a single short reply; costly for anything longer.
- **No VAD at all.** Skips the extra dependency, and the pipeline still technically works —
  Deepgram's own endpointing can detect silence — but there's no acoustic signal for turn-taking
  or barge-in, so the agent can't recognize an interruption mid-sentence the way a VAD-backed
  pipeline can.

None of these are "wrong" in isolation — grouped together, they're exactly the naive default a
team ships when nobody has explicitly engineered for latency yet.

**Why these can only be constructed, not run, here:** every one of these is a real
`DeepgramSTTService` / `CartesiaTTSService` object — constructing them proves the settings are
spelled correctly against the actual installed API (a real, Tier A check); actually driving audio
through them needs a real key and a real transport (Tier B — see the cheat sheet for exactly what
a trainee runs).


In [ ]:
from pipecat.services.deepgram.stt import DeepgramSTTService
from pipecat.services.cartesia.tts import CartesiaTTSService
from pipecat.services.tts_service import TextAggregationMode

# Real service objects, real (if unset) API keys — construction alone doesn't hit the network,
# so this is Tier A: it proves these settings are spelled correctly against the real, installed
# 1.5.0 API surface, without needing DEEPGRAM_API_KEY / CARTESIA_API_KEY to be real.
weak_stt = DeepgramSTTService(
    api_key="placeholder-needs-a-real-key",
    settings=DeepgramSTTService.Settings(interim_results=False),
)

weak_tts = CartesiaTTSService(
    api_key="placeholder-needs-a-real-key",
    settings=CartesiaTTSService.Settings(voice={"mode": "id", "id": "placeholder-voice-id"}),
    text_aggregation_mode=TextAggregationMode.SENTENCE,
)

print("weak_stt.interim_results:", weak_stt._settings.interim_results)
print("weak_tts text_aggregation_mode:", weak_tts._text_aggregation_mode)
print("\nBoth services constructed against the real API — no VAD wired in, no streaming STT.")


### ElevenLabs as an alternative to Deepgram

Deepgram isn't the only STT provider Pipecat wraps natively — the track's tech-stack table lists
**ElevenLabs Scribe v2** alongside Deepgram Nova-3/Flux and AssemblyAI Universal-3 for exactly this
reason: production CX voice stacks pick an STT vendor, they don't get locked into one by the
framework. `pipecat.services.elevenlabs.stt.ElevenLabsSTTService` is the drop-in equivalent of
`DeepgramSTTService` above — same Tier A guarantee (construction alone proves the settings are
spelled correctly against the real, installed API, no key needed to prove that much).

**A real gotcha, not a Deepgram one:** `DeepgramSTTService` above didn't need an HTTP session
passed in — it manages its own connection internally. `ElevenLabsSTTService` does **not**; it
requires an `aiohttp_session` argument explicitly. Skip it and construction fails immediately with
a missing-argument `TypeError`, not a subtle runtime bug three cells later.

**A real, honestly-named limitation, not glossed over:** Step 4 fixes Deepgram's latency by
flipping `interim_results=True`. `ElevenLabsSTTService.InputParams` has no equivalent
streaming-interim-transcript toggle — only `language` and `tag_audio_events`. That's not a gap in
this notebook; it's a genuine difference between the two providers' APIs as wrapped by Pipecat
today, worth knowing before assuming every STT service exposes the same knobs.

**One real number, not a guess — useful directly in the latency table below Step 5:** Pipecat
ships a documented default `ttfs_p99_latency` (time-to-first-speech, p99) estimate per service.
Deepgram's default is `0.35s`; ElevenLabs Scribe's is `2.01s`. That's the library's own
documented estimate, not this notebook's guess — and it's a concrete illustration of exactly the
kind of provider tradeoff the "Voice — STT" row of the tech-stack table is naming when it lists
three providers instead of one.


In [ ]:
import aiohttp
from pipecat.services.elevenlabs.stt import ElevenLabsSTTService

async def _build_elevenlabs_stt():
    # ElevenLabsSTTService needs an aiohttp.ClientSession handed in explicitly -- the real
    # gotcha named above. A real pipeline closes this session on teardown; skipped here since
    # this cell only proves construction, the same Tier A guarantee as weak_stt/weak_tts above.
    session = aiohttp.ClientSession()
    return ElevenLabsSTTService(
        api_key="placeholder-needs-a-real-key",
        aiohttp_session=session,
        sample_rate=16000,
    )

elevenlabs_stt = await _build_elevenlabs_stt()
print("elevenlabs_stt constructed:", type(elevenlabs_stt).__name__)

print("\nPipecat's own documented ttfs_p99_latency defaults (not measured here — quoted from "
      "the library, same honesty rule as the latency table below Step 5):")
print("  Deepgram STT:   0.35s")
print("  ElevenLabs STT:", ElevenLabsSTTService.__init__.__kwdefaults__["ttfs_p99_latency"], "s")


### Step 4 — Fixing it: stream everything, add VAD

The fix mirrors the pattern from every other "weak → fix" cell this week — same tool, same facts,
tighter configuration:

- **`interim_results=True`** — the LLM's context aggregator can start accumulating the user's
  turn as it's transcribed, instead of waiting for one final blob.
- **`text_aggregation_mode=TextAggregationMode.TOKEN`** — TTS starts synthesizing as soon as
  enough tokens exist for a chunk, not a whole sentence. Faster time-to-first-audio, some
  provider-dependent quality tradeoff — Pipecat's own docstring names this tradeoff explicitly,
  it isn't hidden.
- **`SileroVADAnalyzer`**, wired into the transport's params (not shown standalone here — Step 5
  benchmarks it directly). Gives the pipeline a real acoustic signal for turn-taking and
  barge-in, instead of relying on STT-side silence detection alone.

🔍 **Try it yourself:** swap `TOKEN` back to `SENTENCE` on a real deployment and listen for the
difference in how quickly the agent starts speaking after a short answer — the ~200-300ms Pipecat
cites per sentence is very audible on a one-sentence reply, less noticeable on a five-sentence one.


In [ ]:
fixed_stt = DeepgramSTTService(
    api_key="placeholder-needs-a-real-key",
    settings=DeepgramSTTService.Settings(interim_results=True),
)

fixed_tts = CartesiaTTSService(
    api_key="placeholder-needs-a-real-key",
    settings=CartesiaTTSService.Settings(voice={"mode": "id", "id": "placeholder-voice-id"}),
    text_aggregation_mode=TextAggregationMode.TOKEN,
)

fixed_vad = SileroVADAnalyzer(sample_rate=16000, params=VADParams())
fixed_vad.set_sample_rate(16000)  # transports do this automatically; standalone code must do it explicitly

print("fixed_stt.interim_results:", fixed_stt._settings.interim_results)
print("fixed_tts text_aggregation_mode:", fixed_tts._text_aggregation_mode)
print("fixed_vad ready, sample_rate:", fixed_vad.sample_rate)


### Step 5 — The pipeline object graph, and a real VAD benchmark

**Wiring the full pipeline, without a transport.** The canonical Pipecat cascade order is:

```
transport.input() → STT → user context aggregator → LLM → TTS → transport.output() → assistant context aggregator
```

The cell below constructs everything **except** `transport.input()`/`transport.output()` — those
need a real audio source (mic, WebRTC, or a phone line), which this sandbox doesn't have. Building
the rest proves the object graph — STT, the context aggregator pair, the LLM with its tool wired
in, TTS — links together correctly via Pipecat's real `Pipeline([...])` constructor. That's a
genuine Tier A check: a typo in processor order, a missing aggregator, or a malformed tool schema
would fail right here, before a transport is ever involved.

**The VAD benchmark is the one piece of "latency engineering" that's fully real here, not
estimated:** `SileroVADAnalyzer.voice_confidence()` runs a real ONNX model, locally, on the
bundled `claim_status_call.wav` fixture — timing it gives an actual, measured number for one
stage of the budget, not a guess.


In [ ]:
from pipecat.services.anthropic.llm import AnthropicLLMService
from pipecat.processors.aggregators.llm_context import LLMContext
from pipecat.processors.aggregators.llm_response_universal import LLMContextAggregatorPair

fixed_llm = AnthropicLLMService(
    api_key="placeholder-needs-a-real-key",
    settings=AnthropicLLMService.Settings(
        model="claude-sonnet-4-6",
        system_instruction=(
            "You are a voice claims assistant. For ANY question about an existing claim, you "
            "MUST call voice_claim_lookup first and answer only from the returned record. "
            "Your response will be spoken aloud: answer in one flowing spoken sentence, no "
            "bullet points, no markdown, no emoji."
        ),
    ),
)

context = LLMContext(tools=[voice_claim_lookup])
context_aggregator = LLMContextAggregatorPair(context)

# The real canonical order, transport legs omitted (no real audio source in this sandbox):
h1_pipeline = Pipeline([
    fixed_stt,
    context_aggregator.user(),
    fixed_llm,
    fixed_tts,
    context_aggregator.assistant(),
])

print("Pipeline constructed:", len(h1_pipeline.processors), "processors (incl. auto source/sink).")
print("Processor order:", [p.name for p in h1_pipeline.processors])

# --- Real VAD benchmark against the bundled fixture — no key, no network ---
with wave.open("claim_status_call.wav", "rb") as f:
    sr = f.getframerate()
    pcm = f.readframes(f.getnframes())

bench_vad = SileroVADAnalyzer(sample_rate=sr, params=VADParams())
bench_vad.set_sample_rate(sr)
frame_bytes = bench_vad.num_frames_required() * 2

t0 = time.perf_counter()
n_chunks = 0
for i in range(0, len(pcm) - frame_bytes, frame_bytes):
    bench_vad.voice_confidence(pcm[i : i + frame_bytes])
    n_chunks += 1
elapsed = time.perf_counter() - t0

per_chunk_ms = (elapsed / n_chunks) * 1000
chunk_duration_ms = (frame_bytes / 2 / sr) * 1000
print(f"\nVAD processed {n_chunks} chunks of {chunk_duration_ms:.0f}ms audio each in "
      f"{elapsed*1000:.1f}ms total ({per_chunk_ms:.2f}ms/chunk, CPU, this machine).")
print(f"Real-time factor: {chunk_duration_ms / per_chunk_ms:.0f}x "
      "(how many multiples of real-time this VAD can keep up with on this CPU).")


### The latency budget: one real number, the rest honestly estimated

| Stage | Weak config | Fixed config | Source |
|---|---|---|---|
| VAD analysis | ~same | **measured above, this machine** ⬆ | 🟢 real |
| STT time-to-final | full utterance + finalize | first interim ~200-300ms after speech starts | 🟡 Deepgram's published streaming latency, not measured here |
| LLM time-to-first-token | — | provider/model-dependent, typically 300-800ms | 🟡 representative, not measured here |
| TTS time-to-first-audio | wait for full sentence: **+200-300ms/sentence** (Pipecat's own docstring) | token-level: near-immediate after first LLM tokens | 🟢 the +200-300ms figure is Pipecat's own documented cost, not guessed |

**Why this table mixes real and estimated numbers instead of pretending both are equally solid:**
the VAD row and the TTS aggregation-mode row are numbers *this notebook actually produced or
quoted from the library's own documentation* — they're as real as Day 1/2's live Claude calls.
The STT and LLM rows depend on a real network round-trip to a real provider, which this sandbox
cannot make — presenting them as measured would be exactly the "confident, uncited, wrong"
failure Day 1 spent its whole first lab warning about, just applied to a latency claim instead of
a coverage claim. A voice-to-voice budget target (a common one: under ~800ms-1s p50) is a Tier B
exercise — reproduce this table with your own provider's key and real timestamps once you have
one. The ElevenLabs-vs-Deepgram `ttfs_p99_latency` numbers from the callout above belong in this
same table's STT row once you're comparing providers for real, for exactly the same reason —
they're Pipecat's documented defaults, not this notebook's guess.

✅ **Checkpoint — Lab 1 complete.** The pipeline object graph is real and correctly wired
(verified above); the streaming-vs-blocking configuration choices are real Pipecat settings, not
invented ones; the one number that's fully yours (VAD throughput) is measured, not estimated.


---

## Lab 2 — Banking: a fallback that's actually tested, not just declared

**Ship criterion:** an STT layer that keeps a call alive when the primary provider errors out,
demonstrated by an actual simulated failure and an actual observed switch — plus a WER-based gate
that proves the fallback's transcript quality is acceptable, not just "present."

### Step 6 — `ServiceSwitcher`: Pipecat's own failover primitive

Unlike Day 2's Lab 2 (where the idempotency gate was hand-built), this one doesn't need to be —
Pipecat ships `ServiceSwitcher` + `ServiceSwitcherStrategyFailover` for exactly this pattern. Its
own docstring gives the intended shape:

```python
switcher = ServiceSwitcher(services=[primary_stt, backup_stt], strategy_type=ServiceSwitcherStrategyFailover)

@switcher.strategy.event_handler("on_service_switched")
async def on_switched(strategy, service):
    ...
```

**What "weak" looks like here, concretely:** a single STT service, no switcher at all. When it
errors — a dropped websocket, a provider outage — there is no code path that routes around it.
The call doesn't get a wrong answer (Day 1/2's usual failure shape); it gets **no answer**, because
nothing downstream of the dead STT connection ever fires again.


In [ ]:
from pipecat.pipeline.service_switcher import (
    ServiceSwitcher, ServiceSwitcherStrategyFailover, ServiceSwitcherStrategyManual,
)
from pipecat.services.assemblyai.stt import AssemblyAISTTService
from pipecat.frames.frames import ErrorFrame

primary_stt = DeepgramSTTService(api_key="placeholder-needs-a-real-key")
# weak_pipeline_services: just the one STT, no switcher — the actual bug under test.
print("weak setup: single STT service, no fallback path.")
print("If primary_stt errors, there is no other processor in the chain that can pick up "
      "the call — this is dead air, not a wrong answer.")


### Fixing it — wrap both in a `ServiceSwitcher`

`fallback_stt` is a second, independent STT service (AssemblyAI). Wrapping both in a
`ServiceSwitcher` with the `Failover` strategy means: if the active service pushes a non-fatal
`ErrorFrame` upstream, the switcher automatically moves to the next service in the list — no
application code has to notice the failure and react, the framework does it.

**What this cell actually tests, and why it's real:** rather than trying to force a real Deepgram
websocket to fail (needs a real key and a real network fault), the cell drives the
`ServiceSwitcherStrategyFailover` object directly with a real `ErrorFrame` — the exact same object
`ServiceSwitcher` would construct and call internally when a real service errors. This is the same
move as Day 2's LangGraph offline check: swap the thing that needs a live network for a
deterministic stand-in, keep the actual control-flow logic (the strategy class) completely real.


In [ ]:
fallback_stt = AssemblyAISTTService(api_key="placeholder-needs-a-real-key")

switcher = ServiceSwitcher(
    services=[primary_stt, fallback_stt],
    strategy_type=ServiceSwitcherStrategyFailover,
)

switch_events = []
@switcher.strategy.event_handler("on_service_switched")
async def _on_switched(strategy, service):
    switch_events.append(service)

print("Before failure — active service:", type(switcher.strategy.active_service).__name__)
assert switcher.strategy.active_service is primary_stt

# Simulate the exact failure a real dropped websocket / provider 500 would produce upstream —
# a real ErrorFrame, non-fatal, attributed to the currently-active service.
await switcher.strategy.handle_error(ErrorFrame(error="websocket closed unexpectedly", processor=primary_stt))

# Async event handlers (the style Pipecat's own docstring uses) are dispatched via
# asyncio.create_task — fire-and-forget, not awaited inline — so the handler hasn't
# necessarily run yet the instant handle_error() returns. A tick lets it complete before
# we check what it recorded; a real pipeline just keeps running, so this only matters here.
await asyncio.sleep(0.01)

print("After simulated primary failure — active service:", type(switcher.strategy.active_service).__name__)
assert switcher.strategy.active_service is fallback_stt
assert len(switch_events) == 1 and switch_events[0] is fallback_stt

print("\nOK: a real ErrorFrame from the primary really does flip the switcher to the fallback, "
      "and the on_service_switched event really does fire — the call keeps a working STT path.")


### ElevenLabs as a third leg in the same failover chain

The Step 3 callout introduced `ElevenLabsSTTService` as a construction-level alternative to
Deepgram; it fits into this exact `ServiceSwitcher` list the same way `fallback_stt` (AssemblyAI)
just did — no new pattern, one more service in the list, same `Failover` strategy, same
`on_service_switched` event. This is the concrete version of the tech-stack table's "Voice — STT"
row listing three providers: a real production `ServiceSwitcher` chain, not a single vendor with
no way out.


In [ ]:
# Reuses the _build_elevenlabs_stt() helper from the Step 3 callout — same construction,
# same aiohttp_session gotcha, just a second instance for this chain.
elevenlabs_stt_fallback = await _build_elevenlabs_stt()

three_way_switcher = ServiceSwitcher(
    services=[primary_stt, fallback_stt, elevenlabs_stt_fallback],
    strategy_type=ServiceSwitcherStrategyFailover,
)
assert three_way_switcher.strategy.active_service is primary_stt

print("Three-way switcher constructed: Deepgram (primary) -> AssemblyAI -> ElevenLabs.")
print("Active service on construction:", type(three_way_switcher.strategy.active_service).__name__)
print("Same handle_error()/on_service_switched mechanism verified two cells above applies "
      "identically here — adding a provider is a one-line change to the `services=` list, "
      "not a new failover mechanism.")


### Step 7 — Voice eval & QA: is the fallback actually good enough?

Failing over keeps the call alive — it says nothing about whether the **fallback's transcript
quality** is acceptable. A silent accuracy cliff (primary: clean transcripts; fallback: garbled
ones nobody checked) is a real, common failure mode of "we have a fallback" claims. **Word Error
Rate (WER)** is the standard way to quantify it: edit distance between a reference transcript and
what the STT actually produced, normalized by reference length.

**What's real here and what's representative — stated plainly:** `jiwer.wer(reference, hypothesis)`
below is a real WER computation, not a mock — feed it any two strings and it computes a genuine
edit-distance-based score. The *hypothesis* strings (what "the STT produced") are representative
examples standing in for real transcription output, since actually calling Deepgram/AssemblyAI
needs a real key. The **gate logic** — reject a fallback whose WER crosses a threshold — is real
and reusable verbatim once real transcripts are available.


In [ ]:
import jiwer

reference = "my claim id is C L M dash one zero seven seven and I want to know why it was denied"

# Representative hypotheses standing in for real STT output on the same utterance — labeled as
# such, not measured. The WER computation itself is real regardless of where the strings came from.
primary_hypothesis = "my claim id is CLM dash one zero seven seven and I want to know why it was denied"
fallback_hypothesis_good = "my claim id is CLM dash one zero seven seven and I want to know why it was denied"
fallback_hypothesis_degraded = "my claim id is CLM dash one oh seven seven and I want to know why it was denied"

def wer_gate(reference: str, hypothesis: str, max_wer: float = 0.15) -> dict:
    """Real gate logic: compute WER, pass/fail against a threshold. Reusable as-is once the
    hypothesis strings come from a real STT call instead of a representative example."""
    score = jiwer.wer(reference, hypothesis)
    return {"wer": score, "passed": score <= max_wer, "threshold": max_wer}

for label, hyp in [
    ("primary (representative)", primary_hypothesis),
    ("fallback, good quality (representative)", fallback_hypothesis_good),
    ("fallback, degraded quality (representative)", fallback_hypothesis_degraded),
]:
    result = wer_gate(reference, hyp)
    status = "PASS" if result["passed"] else "FAIL"
    print(f"[{status}] {label}: WER={result['wer']:.3f} (threshold={result['threshold']})")

print("\nThe gate itself — jiwer.wer() + the threshold check — is real. Reproduce this cell with "
      "real transcripts from your own Deepgram/AssemblyAI keys (Tier B) to get a real verdict.")


✅ **Checkpoint — Lab 2 complete.** The failover is driven by a real `ErrorFrame` through
Pipecat's actual `ServiceSwitcherStrategyFailover`, not simulated app logic — and the WER gate that
decides whether the fallback is trustworthy is real math with a real threshold, ready for real
transcripts. What's representative is clearly named: the two providers' actual transcription
quality on real audio.


---

## Lab 3 — Telecom: a call flow that can't reach "recording" without consent

**Ship criterion:** a compliant call flow — AI disclosure, a consent prompt, recording gated on
consent, and a replayable audit trail — where the gate is a property of the code, not a hope about
what the LLM remembers to say.

### Step 8 — Telephony: what's real here and what genuinely isn't

Bridging to an actual phone number needs a SIP trunk or a telephony provider account (Twilio,
Telnyx, …) and an actual inbound call — nothing about that can run in a notebook, this one
included. Pipecat's real telephony path (confirmed against the installed 1.5.0 package, not
assumed) is `FastAPIWebsocketTransport` from `pipecat.transports.websocket.fastapi` paired with a
provider serializer — `pipecat.serializers.twilio.TwilioFrameSerializer` or
`pipecat.serializers.telnyx.TelnyxFrameSerializer` both exist in this exact install. That's Tier C:
real code, correct import paths, cannot be exercised without a real trunk and a real inbound call.

**What Lab 3 actually tests instead — and why this is the honest trade, not a workaround:** a
phone bridge is a transport concern — it decides how audio gets to the pipeline. The compliance
requirement (disclose, get consent, gate recording on it, log everything) is a **call-flow logic**
concern, sitting *above* the transport, and every part of that logic runs in plain Python with no
telephony hardware involved at all. Testing it fully here isn't a lesser version of the real thing
— it's testing the actual part that was ever at risk of being wrong.

### Step 9 — Compliance: structural, not conversational

Same discipline as Day 1's `file_claim`: there is no code path from "call connects" to "recording
starts" that skips consent. That's a stronger guarantee than instructing an LLM to "ask for
consent before recording" — a system prompt can be ignored or reordered by a model under the
wrong circumstances; a state machine with no transition for it can't.


In [ ]:
from enum import Enum

class CallState(Enum):
    CONNECTED = "connected"
    DISCLOSED = "disclosed"
    AWAITING_CONSENT = "awaiting_consent"
    CONSENT_GRANTED = "consent_granted"
    CONSENT_DENIED = "consent_denied"
    RECORDING = "recording"
    ENDED = "ended"

audit_log = []   # append-only — this IS the replayable audit trail (Governance row: ISO/IEC
                  # 42001, EU AI Act voice disclosure, GDPR/DPDP all require one of these)

def log_audit(call_id: str, event: str, **details):
    entry = {"call_id": call_id, "event": event, "ts": time.time(), **details}
    audit_log.append(entry)
    return entry


class CompliantCallFlow:
    """Structural gate: recording_active can only become True by passing through
    on_consent_response(granted=True), and every transition is logged. There is no
    other code path to RECORDING."""

    def __init__(self, call_id: str):
        self.call_id = call_id
        self.state = CallState.CONNECTED
        self.recording_active = False

    def on_connect(self):
        assert self.state == CallState.CONNECTED
        log_audit(self.call_id, "disclosure_given",
                   text="This call may be recorded for quality and training purposes.")
        self.state = CallState.DISCLOSED
        log_audit(self.call_id, "consent_requested",
                   text="Do you consent to this call being recorded?")
        self.state = CallState.AWAITING_CONSENT

    def on_consent_response(self, granted: bool):
        assert self.state == CallState.AWAITING_CONSENT
        if granted:
            self.state = CallState.CONSENT_GRANTED
            log_audit(self.call_id, "consent_granted")
            self.recording_active = True
            self.state = CallState.RECORDING
            log_audit(self.call_id, "recording_started")
        else:
            self.state = CallState.CONSENT_DENIED
            log_audit(self.call_id, "consent_denied")
            # No transition to RECORDING exists from here — recording_active stays False
            # for the rest of this call's lifetime, structurally, not by convention.

    def end_call(self):
        if self.recording_active:
            log_audit(self.call_id, "recording_stopped")
        log_audit(self.call_id, "call_ended", final_state=self.state.value)
        self.state = CallState.ENDED


print("CompliantCallFlow defined.")


### The non-compliant version, for contrast

Not a strawman — this is what "the agent starts talking and just starts recording" looks like in
code: no disclosure, no consent gate, recording flips on at connect. Included so the checkpoint
tests below can assert the DIFFERENCE, not just that the compliant version "seems fine."


In [ ]:
class NonCompliantCallFlow:
    """What Step 9 is fixing — recording starts immediately, no disclosure, no consent gate."""
    def __init__(self, call_id: str):
        self.call_id = call_id
        self.recording_active = True   # on by default — the actual bug
        log_audit(call_id, "recording_started")   # no disclosure_given or consent_granted before this

weak_flow = NonCompliantCallFlow("call-weak-1")
weak_events = [e["event"] for e in audit_log if e["call_id"] == "call-weak-1"]
print("Non-compliant call's audit trail:", weak_events)
assert "disclosure_given" not in weak_events
assert "consent_granted" not in weak_events
assert weak_flow.recording_active is True
print("\nConfirmed: recording is active with zero disclosure or consent events in the log — "
      "exactly the non-compliant pattern Step 9's structural gate makes impossible.")


### Checkpoint tests — three real scenarios, driven by a scripted caller

No transport, no audio, no key — this is pure state-machine logic, and it's the part of Lab 3
that was ever actually at risk of a compliance bug. All three assertions are real.


In [ ]:
# Scenario 1 — consent denied: recording must NEVER activate, ever.
flow_denied = CompliantCallFlow("call-1")
flow_denied.on_connect()
flow_denied.on_consent_response(granted=False)
flow_denied.end_call()

denied_events = [e["event"] for e in audit_log if e["call_id"] == "call-1"]
print("Consent-denied call events:", denied_events)
assert denied_events == [
    "disclosure_given", "consent_requested", "consent_denied", "call_ended",
]
assert flow_denied.recording_active is False
assert "recording_started" not in denied_events

# Scenario 2 — consent granted: disclosure and consent must precede recording, in order.
flow_granted = CompliantCallFlow("call-2")
flow_granted.on_connect()
flow_granted.on_consent_response(granted=True)
flow_granted.end_call()

granted_events = [e["event"] for e in audit_log if e["call_id"] == "call-2"]
print("Consent-granted call events:", granted_events)
assert granted_events == [
    "disclosure_given", "consent_requested", "consent_granted",
    "recording_started", "recording_stopped", "call_ended",
]
assert granted_events.index("disclosure_given") < granted_events.index("recording_started")
assert granted_events.index("consent_granted") < granted_events.index("recording_started")

# Scenario 3 — a structural attempt to skip the gate: calling on_consent_response() before
# on_connect() must fail loudly, not silently activate recording.
flow_bypass = CompliantCallFlow("call-3")
try:
    flow_bypass.on_consent_response(granted=True)
    raise RuntimeError("bypass should have been rejected but wasn't")
except AssertionError:
    print("Attempt to grant consent before disclosure was correctly rejected (AssertionError).")
assert flow_bypass.recording_active is False

print("\nOK: all three scenarios hold — denial never records, consent always precedes "
      "recording in the log, and the state machine rejects an out-of-order call entirely.")


### Replayability — the audit trail has to reconstruct the call, not just list events

A "replayable audit trail" (the Governance row's actual requirement) means a third party can
reconstruct what happened from the log alone, without trusting anything the system currently
reports about itself — the same principle as Day 2 Lab 3's QA hook: verify from outside, don't
trust self-report.


In [ ]:
def replay_final_state(call_id: str) -> dict:
    """Reconstructs whether a call was disclosed, consented, and recorded — purely from the
    audit log, independent of any live CompliantCallFlow object. This is the actual test of
    'replayable': can a compliance reviewer answer these questions from the log alone, months
    later, with the process that generated it long gone."""
    events = [e["event"] for e in audit_log if e["call_id"] == call_id]
    return {
        "disclosed": "disclosure_given" in events,
        "consent_granted": "consent_granted" in events,
        "recorded": "recording_started" in events,
        "ended_cleanly": events[-1] == "call_ended" if events else False,
    }

replay_1 = replay_final_state("call-1")
replay_2 = replay_final_state("call-2")
print("Replay of call-1 (consent denied):", replay_1)
print("Replay of call-2 (consent granted):", replay_2)

assert replay_1 == {"disclosed": True, "consent_granted": False, "recorded": False, "ended_cleanly": True}
assert replay_2 == {"disclosed": True, "consent_granted": True, "recorded": True, "ended_cleanly": True}

print("\nOK: the audit log alone — no live object, no self-report — correctly answers "
      "'was this call compliant' for both scenarios.")


✅ **Checkpoint — Lab 3 complete.** The compliance gate is structural (verified above — three
real scenarios, including a rejected bypass attempt) and the audit trail is genuinely replayable
(verified independently of the live call-flow object). The telephony bridge that would carry a
real call to this flow is real, correctly-imported Pipecat code — and honestly marked as
untestable here, because it is.


---

## Closing out

Three labs, one shared discipline underneath the surface topics (assembly, reliability,
compliance): **a latency number, a fallback, and a consent gate are only real if something outside
the agent's own good intentions verified them.** Lab 1's latency table names exactly which numbers
were measured versus estimated, instead of blurring them into one confident figure. Lab 2's
fallback is proven by a real `ErrorFrame`, not by trusting that "it should work." Lab 3's
compliance gate is structural — no code path around it — and its audit trail was checked by
replaying the log alone, the same "verify from outside, don't trust self-report" discipline as
Day 2's QA hook.

The corollary, worth saying once directly: everything in this notebook that could be verified
without a real key or a real phone line, was — pipeline wiring, VAD throughput, the failover
event, the WER gate math, all three compliance scenarios including a rejected bypass. Everything
that couldn't was named as such, not quietly assumed. That's the actual skill "voice eval & QA"
is teaching, one level up: knowing which of your own claims about a system you've actually
checked.

---

## Ship rubric

| Checkpoint | Pass criteria |
|---|---|
| Lab 1 | `Pipeline([...])` constructs with the correct canonical processor order (STT → user aggregator → LLM → TTS → assistant aggregator) |
| Lab 1 | The weak vs. fixed STT/TTS settings are real, correctly-named Pipecat config fields, not invented ones |
| Lab 1 | The VAD benchmark produces a real measured per-chunk time and real-time factor on your machine |
| Lab 2 | A real `ErrorFrame` on the primary STT triggers a real `on_service_switched` event to the fallback |
| Lab 2 | The WER gate correctly passes a clean transcript and fails a degraded one against the stated threshold |
| Lab 3 | Consent-denied calls never set `recording_active = True`, verified by the log, not by inspection |
| Lab 3 | Consent-granted calls show disclosure and consent preceding `recording_started`, in that order, in the log |
| Lab 3 | The audit trail reconstructs a call's compliance status by replay alone, independent of the live object |

## Known-breakage cheat sheet

| Symptom | Likely cause | Fix |
|---|---|---|
| `ImportError` on `PipelineTask` / `PipelineRunner` | Following an older tutorial — both are deprecated in 1.5.0 | Use `PipelineWorker` (`pipecat.pipeline.worker`) and `WorkerRunner` (`pipecat.workers.runner`) |
| `SileroVADAnalyzer.voice_confidence()` errors with "Supported sampling rates" | `set_sample_rate()` was never called — the constructor's `sample_rate=` argument doesn't take effect on its own | Call `vad.set_sample_rate(sr)` explicitly after construction (a transport does this automatically in a real pipeline) |
| `FunctionCallParams(...)` raises a missing-argument `TypeError` | `pipeline_worker` is a required field, easy to miss from an older example | Pass `pipeline_worker=None` for offline tests, or the real worker in a running pipeline |
| `on_service_switched` handler never seems to fire when checked immediately after `handle_error()` | Async event handlers are dispatched via `asyncio.create_task` — fire-and-forget, not awaited inline | `await asyncio.sleep(0)` (or just let a real pipeline keep running) before checking what the handler recorded |
| `ElevenLabsSTTService(...)` raises a missing-argument `TypeError` for `aiohttp_session` | Unlike `DeepgramSTTService`, ElevenLabs' service doesn't manage its own HTTP session | Pass `aiohttp_session=aiohttp.ClientSession()` explicitly (see the Step 3 ElevenLabs callout) |
| Auth error on `AnthropicLLMService` / `DeepgramSTTService` / `CartesiaTTSService` even though the rest of the notebook worked | These call their provider's API directly and need a real key — the Claude Code CLI login from Day 1/2 doesn't cover them | Set a real `ANTHROPIC_API_KEY` / `DEEPGRAM_API_KEY` / `CARTESIA_API_KEY` / `ELEVENLABS_API_KEY` |
| A Pipecat snippet from an old blog post / your own memory doesn't match the installed API | Pipecat moves fast; its own docs call stale APIs the #1 failure mode for anyone writing Pipecat code | Check the installed package's source directly (`import pipecat, os; os.path.dirname(pipecat.__file__)`) before trusting any remembered signature |

---

## Appendix — the same agent in LiveKit Agents

Lab 1 was built on Pipecat, on purpose — a literal Python list of processors makes every teaching
beat a visible object change. This section exists for **framework literacy**, same spirit as
Day 2's LangGraph/CrewAI appendix: the same claim-lookup voice agent, rebuilt on
[LiveKit Agents](https://docs.livekit.io/agents/) (1.6.6, confirmed against the installed
package), because "voice orchestration" is a family of frameworks, not one shape. Optional — skip
it if you're short on time.

**The structural difference, stated plainly:** Pipecat's pipeline is a **list you assemble** —
`Pipeline([stt, agg, llm, tts, agg])`, order and composition fully explicit. LiveKit's `Agent` is
a **single object you configure** — `Agent(instructions=..., stt=..., vad=..., llm=..., tts=...)`
— the pipeline order is implicit, owned by the framework. Same five ingredients, opposite default
for how much of the wiring you write yourself.

```bash
pip install livekit-agents livekit-plugins-deepgram livekit-plugins-cartesia \
            livekit-plugins-anthropic livekit-plugins-silero livekit-plugins-elevenlabs
```


In [ ]:
from livekit.agents import Agent, AgentSession, function_tool
from livekit.plugins import deepgram as lk_deepgram, cartesia as lk_cartesia
from livekit.plugins import anthropic as lk_anthropic, silero as lk_silero
from livekit.plugins import elevenlabs as lk_elevenlabs

# Same @tool-as-plain-function idea as every framework this week — name, type hints, and
# docstring become the schema. LiveKit's version returns the result directly (a plain
# return value) rather than Pipecat's explicit params.result_callback(...) call.
@function_tool
async def lk_claim_lookup(claim_id: str) -> dict:
    """Look up the status of an existing insurance claim by its claim id."""
    record = claims_db.get(claim_id)
    return record if record else {"error": f"No claim found with id {claim_id}"}

# Real service objects against the real installed 1.6.6 API — construction alone doesn't
# hit the network for STT/TTS/LLM (same Tier A logic as the Pipecat services earlier).
lk_stt = lk_deepgram.STT(api_key="placeholder-needs-a-real-key")
lk_tts = lk_cartesia.TTS(api_key="placeholder-needs-a-real-key")
lk_llm = lk_anthropic.LLM(model="claude-sonnet-4-6", api_key="placeholder-needs-a-real-key")

# ElevenLabs, as an alternative to Deepgram, here too — same provider as the Pipecat callout,
# a DIFFERENT ergonomics story under this framework: LiveKit's plugin makes the HTTP session
# optional (it manages its own by default if none is passed), unlike Pipecat's version, which
# requires one explicitly. Same vendor, same STT capability, one fewer required argument —
# worth noticing that "the same provider" doesn't mean "the same construction gotchas" once a
# different orchestration framework is wrapping it.
lk_elevenlabs_stt = lk_elevenlabs.STT(api_key="placeholder-needs-a-real-key")
print("lk_elevenlabs_stt constructed:", type(lk_elevenlabs_stt).__name__,
      "— a drop-in alternative to lk_stt below, no aiohttp session required this time.")

# Unlike the STT/TTS/LLM constructors above, VAD.load() is NOT just object construction —
# it actually loads a real Silero model, no key required. This one genuinely runs, right now.
lk_vad = lk_silero.VAD.load()
print("Silero VAD loaded for real:", type(lk_vad).__name__)

# All five ingredients as PARAMETERS to one Agent object — contrast with Pipecat's explicit
# Pipeline([...]) list. The pipeline order (STT -> LLM -> TTS, VAD gating turns) is owned by
# the framework here, not spelled out by you. stt=lk_stt below; swap in lk_elevenlabs_stt for
# the ElevenLabs alternative — same Agent(...) shape either way.
lk_agent = Agent(
    instructions=(
        "You are a voice claims assistant. For ANY question about an existing claim, you "
        "MUST call lk_claim_lookup first and answer only from the returned record. Your "
        "response will be spoken aloud: answer in one flowing spoken sentence, no bullet "
        "points, no markdown, no emoji."
    ),
    tools=[lk_claim_lookup],
    stt=lk_stt,
    vad=lk_vad,
    llm=lk_llm,
    tts=lk_tts,
)

lk_session = AgentSession()

print("\nAgent and AgentSession constructed against the real, installed livekit-agents API.")
print("Actually running a turn needs a real LiveKit room connection (rtc.Room().connect(url, "
      "token)) for real audio in/out. LiveKit's own in-process test helper "
      "(testing.fake_job_context) does NOT require one, though — called with no `room` "
      "argument it builds an unconnected stub Room internally, so context construction "
      "alone is testable without a live connection. Tier C applies to a genuine "
      "end-to-end audio turn, not to session/context construction — named precisely "
      "rather than lumped together.")


**The honest limit of this appendix, stated the same way Day 2 named CrewAI's:** everything
above is real, verified construction — including a Silero VAD model that genuinely loaded, not
just imported cleanly. What's not verified here is a running end-to-end audio turn: that needs a
real, connected `Room` (audio actually flowing in and out). Context/session construction alone
does not — LiveKit's own in-process test helper, `livekit.agents.testing.fake_job_context`, builds
an unconnected stub `Room` internally when called with no `room` argument, so it's usable without
a live connection. Pipecat's `Pipeline` still goes further (processors link as plain objects with
no session/context concept to construct at all), but the gap between the two frameworks here is
narrower than "needs a connection vs. doesn't" — worth stating precisely rather than rounding
the difference up to a hard requirement neither library actually has at this layer.

✅ **Appendix complete.** Same voice agent, same tool, two structurally different frameworks —
Pipecat's explicit pipeline list, LiveKit's configured single object — both verified as far as
this sandbox honestly allows. ElevenLabs slotted into both as a real, constructed alternative to
Deepgram wherever Deepgram appeared — Step 3 (primary STT), Lab 2 (a third failover leg), and here
(LiveKit's `Agent`) — same provider, three different frameworks, three different sets of
construction gotchas, one consistent lesson: "the tech-stack table lists more than one vendor per
row" is not a footnote, it's a real, swappable object every time.
